# 04 — NeuralFP Training (pfann, Dry Run)

Trainiert das pfann-Modell (Chang et al. 2021, contrastive learning) für **2 Epochen**
auf `dry_train` (200 Songs) mit MUSAN-Rauschaugmentation.
Room-Impulse-Response-Augmentation ist im Dry Run deaktiviert (keine IR-Daten).

**Portabilität:** Alle .txt-Listen werden mit Pfaden **relativ zu PROJECT_ROOT**
geschrieben. Die Konfigurationsschlüssel `music_dir` und `noise.dir` werden auf
`str(PROJECT_ROOT)` gesetzt, das zur Laufzeit aufgelöst wird. Das Notebook
läuft dadurch unverändert auf Mac, Vertex oder jedem anderen Rechner, sofern
die Daten relativ zur gleichen Verzeichnisstruktur vorliegen.

**Abhängigkeiten:** NB 00 (`data/partitions/dry_train.json`, `dry_ref.json`, `musan_split.json`)
**Ausgabe:**
- `pfann/configs/dry_run.json`
- `pfann/lists/dry_train.txt`, `dry_validate.txt`, `dry_ref.txt`, `musan_train.txt`
- `checkpoints/dry_run/pfann_model/model.pt`


In [ ]:
# ── RUN MODE CONFIG ──────────────────────────────────────────────────────────
# Switch between dry run and live run by editing src/run_config.py.
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))
from run_config import cfg
RUN_MODE = cfg.run_mode
print(f"RUN_MODE = {RUN_MODE!r}")

## 1. Imports und Pfade

In [ ]:
import os, sys, json, re, subprocess, random, time
from pathlib import Path

PROJECT_ROOT = Path("/home/jupyter/liverun/").resolve()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import matplotlib.pyplot as plt

from run_config import cfg
RUN_MODE = cfg.run_mode

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FMA_DIR        = PROJECT_ROOT / "data" / "fma_medium"
MUSAN_DIR      = PROJECT_ROOT / "data" / "musan"
PARTITIONS_DIR = PROJECT_ROOT / "data" / "partitions"
PFANN_DIR      = PROJECT_ROOT / "src" / "pfann"
PFANN_LISTS    = PFANN_DIR / "lists"
PFANN_CONFIGS  = PFANN_DIR / "configs"

# ── Mode-dependent paths ──────────────────────────────────────────────────────
CHECKPOINTS = PROJECT_ROOT / "checkpoints" / cfg.checkpoints_subdir
RESULTS_DIR = PROJECT_ROOT / "results" / cfg.results_subdir
PFANN_CONFIG_NAME = cfg.pfann_config_name     # "dry_run" or "live_run"
LIST_PREFIX       = cfg.pfann_list_prefix     # "dry" or "live"
TRAIN_KEY         = cfg.train_key             # "dry_train" or "live_train"
REF_KEY           = cfg.ref_key               # "dry_ref" or "live_ref"

PFANN_LISTS.mkdir(parents=True, exist_ok=True)
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"RUN_MODE      = {RUN_MODE!r}")
print(f"TRAIN_KEY     = {TRAIN_KEY!r}  (batch_size={cfg.batch_size}, epochs={cfg.epochs})")
print(f"CHECKPOINTS   = {CHECKPOINTS}")
print("PROJECT_ROOT:", PROJECT_ROOT)

## 2. Voraussetzungen prüfen

NB 00 muss `dry_train.json`, `dry_ref.json` und `musan_split.json`
in `data/partitions/` hinterlegt haben.


In [ ]:
from utils import load_partitions, assert_disjoint, export_pfann_list, \
                  write_path_list, reanchor_paths

required = [
    PARTITIONS_DIR / f"{TRAIN_KEY}.json",
    PARTITIONS_DIR / f"{REF_KEY}.json",
    PARTITIONS_DIR / "musan_split.json",
]
for p in required:
    assert p.exists(), f"Voraussetzung fehlt: {p} — NB 00 ausführen!"
    print(f"  OK: {p.name}")
print("Alle Voraussetzungen erfüllt. ✓")

## 3. Partitionen und MUSAN-Split laden

MUSAN-Pfade aus `musan_split.json` können von einer anderen Maschine stammen
(absolute Pfade). `reanchor_paths()` ersetzt den alten Präfix durch das
aktuelle `MUSAN_DIR` — identisch bei lokalem Lauf, korrigiert bei Maschinenübernahme.


In [ ]:
partitions    = load_partitions(PARTITIONS_DIR)
train_ids     = partitions[TRAIN_KEY]
ref_ids       = partitions[REF_KEY]

with open(PARTITIONS_DIR / "musan_split.json") as fh:
    musan_split = json.load(fh)

musan_train_files = reanchor_paths(musan_split["train"], MUSAN_DIR)

print(f"{TRAIN_KEY}     : {len(train_ids)} Tracks")
print(f"{REF_KEY}       : {len(ref_ids)}  Tracks (Validierungsset)")
print(f"musan_train    : {len(musan_train_files)} Dateien")
print(f"  Beispiel: {musan_train_files[0]}")

assert_disjoint(train_ids, ref_ids)

## 4. pfann-Listen exportieren (relative Pfade)

Alle .txt-Listen werden mit Pfaden **relativ zu PROJECT_ROOT** geschrieben.
`music_dir` und `noise.dir` in `dry_run.json` werden auf `str(PROJECT_ROOT)`
gesetzt, das zur Laufzeit aufgelöst wird. Damit funktionieren die Listen
auf jeder Maschine ohne Änderung.

pfann liest Audio-Listen intern als CSV und überspringt die erste Zeile als
Header — für den Dry Run (200 Trainingstracks) ist der Verlust eines Tracks akzeptabel.


In [ ]:
TRAIN_TXT    = PFANN_LISTS / f"{LIST_PREFIX}_train.txt"
VALIDATE_TXT = PFANN_LISTS / f"{LIST_PREFIX}_validate.txt"
REF_TXT      = PFANN_LISTS / f"{LIST_PREFIX}_ref.txt"
MUSAN_TXT    = PFANN_LISTS / "musan_train.txt"

export_pfann_list(train_ids, FMA_DIR, TRAIN_TXT)
export_pfann_list(ref_ids,   FMA_DIR, REF_TXT)
export_pfann_list(ref_ids,   FMA_DIR, VALIDATE_TXT)
write_path_list(musan_train_files, MUSAN_TXT)

with open(TRAIN_TXT) as f:
    sample = f.readlines()[:3]
print(f"{TRAIN_TXT.name} (erste 3 Zeilen):")
for l in sample:
    print(f"  {l.rstrip()}")

## 5. dry_run.json anlegen

`music_dir` und `noise.dir` zeigen auf PROJECT_ROOT. Kombiniert mit den
relativen Pfaden in den .txt-Listen ergibt pfann daraus die korrekten
absoluten Dateipfade: `os.path.join(str(PROJECT_ROOT), "data/fma_medium/...")`.

`batch_size=32` (statt 120): ~200 Songs × ~9.5 Segmente/Song ≈ 1 900 Segmente
→ ~59 Batches/Epoche. Mit 120 wären es nur ~15 — zu wenig für den Loss-Plot.


In [ ]:
config_path = PFANN_CONFIGS / f"{PFANN_CONFIG_NAME}.json"

run_config_dict = {
    "train_csv":    str(TRAIN_TXT),
    "validate_csv": str(VALIDATE_TXT),
    "music_dir":    str(PROJECT_ROOT),
    "model_dir":    str(CHECKPOINTS / "pfann_model"),
    "cache_dir":    str(CHECKPOINTS / "pfann_cache"),
    "batch_size":   cfg.batch_size,
    "shuffle_size": None,
    "fftconv_n":    32768,
    "sample_rate":  8000,
    "stft_n":       1024,
    "stft_hop":     256,
    "n_mels":       256,
    "f_min":        300,
    "f_max":        4000,
    "segment_size": 1,
    "hop_size":     0.5,
    "time_offset":  1.2,
    "pad_start":    0,
    "epoch":        cfg.epochs,
    "lr":           1e-4,
    "tau":          0.05,
    "noise": {
        "train":    str(MUSAN_TXT),
        "validate": None,
        "dir":      str(PROJECT_ROOT),
        "snr_max":  10,
        "snr_min":  0,
    },
    "micirp": {"train": None, "validate": None, "dir": "", "length": 0.5},
    "air":    {"train": None, "validate": None, "dir": "", "length": 1},
    "cutout_min": 0.1,
    "cutout_max": 0.5,
    "model": {
        "d": 128, "h": 1024, "u": 32, "fuller": True, "conv_activation": "ReLU",
    },
    "indexer": {
        "index_factory": "IVF200,PQ64x8np", "top_k": 100, "frame_shift_mul": 1,
    },
}

with open(config_path, "w", encoding="utf-8") as fh:
    json.dump(run_config_dict, fh, indent=2)
print(f"Geschrieben: {config_path}")
print(f"  batch_size = {run_config_dict['batch_size']}")
print(f"  epoch      = {run_config_dict['epoch']}")
print(f"  music_dir  = {run_config_dict['music_dir']}")
print(f"  noise.dir  = {run_config_dict['noise']['dir']}")

## 6. NeuralFP-Training starten (2 Epochen)

`pfann/train.py` als Subprocess mit `check=True`.
`cwd=src/pfann/` ist notwendig — train.py verwendet relative Imports
(`from model import FpNetwork`).

Stdout wird komplett gecaptured; Loss-Werte erscheinen nach Abschluss.

**Hinweis:** pfann erfordert CUDA. Auf macOS ohne CUDA schlägt dieser
Schritt fehl.


In [ ]:
import sys
python_bin = sys.executable
train_cmd  = [
    python_bin, "train.py",
    "--params", str(config_path),
    "-w", "4",
]
print("Kommando:", " ".join(train_cmd))
print(f"cwd: {PFANN_DIR}")

t0 = time.time()
try:
    train_result = subprocess.run(
        train_cmd,
        cwd=str(PFANN_DIR),
        capture_output=True,
        text=True,
        check=True,
    )
except subprocess.CalledProcessError as e:
    print("--- STDOUT ---")
    print(e.stdout[-3000:])
    print("--- STDERR ---")
    print(e.stderr[-3000:])
    raise
train_duration_s = time.time() - t0
print(f"Training abgeschlossen in {train_duration_s / 60:.1f} Min.")
print("--- stdout (letzte 60 Zeilen) ---")
stdout_lines = train_result.stdout.strip().splitlines()
print("\n".join(stdout_lines[-60:]))

## 7. Trainings-Loss plotten

pfann schreibt pro Epoche `loss: X.XXXXXX` nach stdout.
Der Contrastive Loss (NT-Xent) muss von Epoche 1 zu Epoche 2 sinken.


In [ ]:
loss_pattern = re.compile(r'^loss:\s+([0-9.eE+\-]+)', re.MULTILINE)
losses = [float(m) for m in loss_pattern.findall(train_result.stdout)]

if not losses:
    print("WARNUNG: Keine 'loss:'-Zeilen im stdout gefunden.")
    print("Vollständiger stdout:")
    print(train_result.stdout)
else:
    epochs = list(range(1, len(losses) + 1))
    print(f"Loss-Werte pro Epoche: {losses}")

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(epochs, losses, marker="o", linewidth=2, color="steelblue")
    ax.set_xlabel("Epoche")
    ax.set_ylabel("Contrastive Loss (NT-Xent)")
    ax.set_title("NeuralFP Training Loss — Dry Run (2 Epochen)")
    ax.set_xticks(epochs)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plot_path = RESULTS_DIR / "neuralFP_train_loss.png"
    plt.savefig(plot_path, dpi=150)
    plt.show()
    print(f"Plot gespeichert: {plot_path}")

    if len(losses) >= 2:
        trend = "sinkt ✓" if losses[-1] < losses[0] else "WARNUNG: steigt!"
        print(f"Loss {losses[0]:.4f} → {losses[-1]:.4f}  ({trend})")


## 8. Sanity Check

Prüft ob pfann `model.pt` erzeugt hat.
Ohne `model.pt` schlägt builder.py in NB 05 fehl.


In [ ]:
model_dir = CHECKPOINTS / "pfann_model"
model_pt  = model_dir / "model.pt"

assert model_pt.exists(), (
    f"model.pt fehlt: {model_pt}\n"
    "Training fehlgeschlagen — stderr:\n"
    + train_result.stderr[-2000:]
)
model_size_mb = model_pt.stat().st_size / (1024 ** 2)
print(f"model.pt: {model_pt}  ({model_size_mb:.1f} MB) ✓")

ckpt_files = sorted(model_dir.glob("*.ckpt"))
print(f"Checkpoint-Dateien: {len(ckpt_files)}")
for f in ckpt_files:
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

print("\n=== Zusammenfassung ===")
print(f"RUN_MODE       : {RUN_MODE!r}")
print(f"train_ids      : {len(train_ids)} Tracks  ({TRAIN_KEY})")
print(f"ref_ids        : {len(ref_ids)}  Tracks  ({REF_KEY})")
print(f"musan_train    : {len(musan_train_files)} Noise-Dateien")
print(f"batch_size     : {run_config_dict['batch_size']}")
print(f"epoch          : {run_config_dict['epoch']}")
print(f"training_time  : {train_duration_s / 60:.1f} Min")
print(f"model_dir      : {model_dir}")